In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# 1. Load and Sort
df = pd.read_csv('financial_regression.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

In [3]:
print("Min date:", df["date"].min())
print("Max date:", df["date"].max())
print("Date range:", df["date"].max() - df["date"].min())

Min date: 2010-01-14 00:00:00
Max date: 2024-10-23 00:00:00
Date range: 5396 days 00:00:00


In [2]:
# 2. Drop non-trading days for gold
df_clean = df.dropna(subset=['gold close']).reset_index(drop=True)

# 3. Compute intra-day & lagged features
df_clean['Daily_Return'] = (df_clean['gold close'] - df_clean['gold open']) / df_clean['gold open']

In [4]:
# 4. Compute 1-day market log returns (standardizing names with underscores)
market_price_cols = [
    'gold close', 'silver close', 'oil close', 
    'sp500 close', 'nasdaq close', 'platinum close', 'palladium close'
]

for col in market_price_cols:
    clean_name = col.replace(' ', '_')
    df_clean[f'{clean_name}_log_ret'] = np.log(df_clean[col] / df_clean[col].shift(1))

# Volume change
df_clean['gold_volume_change'] = np.log((df_clean['gold volume'] + 1) / (df_clean['gold volume'].shift(1) + 1))

In [5]:
# 5. Cyclical Calendar Features
df_clean["DayOfWeek"] = df_clean["date"].dt.dayofweek
df_clean["Month"] = df_clean["date"].dt.month
df_clean["DayOfYear"] = df_clean["date"].dt.dayofyear
df_clean["IsMonthEnd"] = df_clean["date"].dt.is_month_end.astype(int)

df_clean['Month_sin'] = np.sin(2 * np.pi * df_clean['Month'] / 12)
df_clean['Month_cos'] = np.cos(2 * np.pi * df_clean['Month'] / 12)
df_clean['DayOfYear_sin'] = np.sin(2 * np.pi * df_clean['DayOfYear'] / 365.25)
df_clean['DayOfYear_cos'] = np.cos(2 * np.pi * df_clean['DayOfYear'] / 365.25)

In [6]:
df_clean['Target_Log_Return'] = np.log(df_clean['gold close'].shift(-1) / df_clean['gold close'])
df_clean["Target_Next_Close"] = df_clean["gold close"].shift(-1)

In [7]:
# 7. Define Feature Set
numeric_features = [
    'Daily_Return',
    'gold_close_log_ret',
    'gold_volume_change',
    'silver_close_log_ret',
    'oil_close_log_ret',
    'sp500_close_log_ret',
    'nasdaq_close_log_ret',
    'platinum_close_log_ret',
    'palladium_close_log_ret',
    'Month_sin',
    'Month_cos',
    'DayOfYear_sin',
    'DayOfYear_cos'
]

categorical_features = ['DayOfWeek', 'IsMonthEnd']
features = numeric_features + categorical_features
target = 'Target_Log_Return'

In [8]:
# 8. Final Dropna across ALL features and target
df_model = df_clean.dropna(subset=features + [target]).reset_index(drop=True)

# Verification
print("Null count across selected features:\n", df_model[features + [target]].isna().sum())

Null count across selected features:
 Daily_Return               0
gold_close_log_ret         0
gold_volume_change         0
silver_close_log_ret       0
oil_close_log_ret          0
sp500_close_log_ret        0
nasdaq_close_log_ret       0
platinum_close_log_ret     0
palladium_close_log_ret    0
Month_sin                  0
Month_cos                  0
DayOfYear_sin              0
DayOfYear_cos              0
DayOfWeek                  0
IsMonthEnd                 0
Target_Log_Return          0
dtype: int64


In [11]:
# ---------------------------------------------------------
# 9. Chronological Train / Test Split
# ---------------------------------------------------------
split_index = int(len(df_model) * 0.80)

train = df_model.iloc[:split_index].copy()
test = df_model.iloc[split_index:].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# ---------------------------------------------------------
# 10. Pipeline Setup & TimeSeriesSplit Cross-Validation
# ---------------------------------------------------------
numeric_pipeline = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    [
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ]
)

ridge_pipeline = Pipeline(
    [('preprocessor', preprocessor), ('model', Ridge())]
)

# Expand search grid to evaluate lower penalties suited for stationary returns
ridge_params = {
    'model__alpha': [
        0.0001,
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
        1000.0,
        10000.0,
    ]
}

tscv = TimeSeriesSplit(n_splits=5)

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_params,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

# ---------------------------------------------------------
# 6. Fit & Display CV Results
# ---------------------------------------------------------
ridge_search.fit(X_train, y_train)

best_alpha = ridge_search.best_params_['model__alpha']
best_neg_mse = ridge_search.best_score_
best_rmse = np.sqrt(-best_neg_mse)

print(f'Best Ridge Alpha: {best_alpha}')
print(f'Best CV Neg MSE: {best_neg_mse:.6e}')
print(f'Best CV RMSE:    {best_rmse:.6f} (or {best_rmse * 100:.4f}% daily log return error)')

Best Ridge Alpha: 10000.0
Best CV Neg MSE: -9.149179e-05
Best CV RMSE:    0.009565 (or 0.9565% daily log return error)


In [12]:
# Get feature names after OneHotEncoder preprocessing
preprocessor = ridge_search.best_estimator_['preprocessor']
feature_names = preprocessor.get_feature_names_out()

# Extract coefficients from the fitted Ridge model
coefs = ridge_search.best_estimator_['model'].coef_

coef_df = (
    pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
    .sort_values(by='coefficient', key=abs, ascending=False)
    .reset_index(drop=True)
)

print('Top Ridge Coefficients:')
print(coef_df.head(10))
print('\nIntercept:', ridge_search.best_estimator_['model'].intercept_)

Top Ridge Coefficients:
                         feature  coefficient
0    numeric__gold_volume_change    -0.000065
1     numeric__oil_close_log_ret    -0.000065
2       categorical__DayOfWeek_3     0.000046
3   numeric__sp500_close_log_ret     0.000035
4       categorical__DayOfWeek_1    -0.000029
5         numeric__DayOfYear_sin     0.000021
6  numeric__silver_close_log_ret    -0.000020
7      categorical__IsMonthEnd_0    -0.000018
8      categorical__IsMonthEnd_1     0.000018
9         numeric__DayOfYear_cos    -0.000018

Intercept: 0.00016085460846804208


In [13]:
from sklearn.metrics import mean_squared_error, r2_score

# 1. Model Predictions
y_pred_model = ridge_search.predict(X_test)

# 2. Naive Zero-Return Baseline Predictions
y_pred_baseline = np.zeros_like(y_test)

# 3. Compute Metrics
model_mse = mean_squared_error(y_test, y_pred_model)
baseline_mse = mean_squared_error(y_test, y_pred_baseline)

model_r2 = r2_score(y_test, y_pred_model)
baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f'Model Test MSE:    {model_mse:.6e}')
print(f'Baseline Test MSE: {baseline_mse:.6e}')
print(f'Model Test R2:     {model_r2:.5f}')
print(f'Baseline Test R2:  {baseline_r2:.5f}')

Model Test MSE:    7.998416e-05
Baseline Test MSE: 8.013399e-05
Model Test R2:     -0.00157
Baseline Test R2:  -0.00344


In [14]:
import numpy as np
import pandas as pd

# Assume df_clean contains market price columns and date sorting
# ---------------------------------------------------------
# 1. Multi-Day Momentum Features (2-day, 5-day, 10-day returns)
# ---------------------------------------------------------
lags = [1, 2, 5, 10]
target_assets = [
    'gold close',
    'silver close',
    'oil close',
    'sp500 close',
    'nasdaq close',
]

for asset in target_assets:
    col_name = asset.replace(' ', '_')
    for lag in lags:
        # Multi-day log return over 'lag' days
        df_clean[f'{col_name}_log_ret_{lag}d'] = np.log(
            df_clean[asset] / df_clean[asset].shift(lag)
        )

# ---------------------------------------------------------
# 2. Rolling Volatility & Moving Average Divergence
# ---------------------------------------------------------
# 1-day log return base
gold_ret = df_clean['gold_close_log_ret_1d']

# Volatility over 5-day and 20-day rolling windows
df_clean['gold_volatility_5d'] = gold_ret.rolling(window=5).std()
df_clean['gold_volatility_20d'] = gold_ret.rolling(window=20).std()

# Price distance from 20-day Moving Average (Trend Distance)
gold_sma20 = df_clean['gold close'].rolling(window=20).mean()
df_clean['gold_sma20_dist'] = np.log(df_clean['gold close'] / gold_sma20)

# ---------------------------------------------------------
# 3. Inter-Market Cross Ratios (Gold/Silver & Gold/Oil)
# ---------------------------------------------------------
df_clean['gold_silver_ratio'] = (
    df_clean['gold close'] / df_clean['silver close']
)
df_clean['gold_silver_ratio_chg'] = np.log(
    df_clean['gold_silver_ratio'] / df_clean['gold_silver_ratio'].shift(1)
)

df_clean['gold_oil_ratio'] = df_clean['gold close'] / df_clean['oil close']
df_clean['gold_oil_ratio_chg'] = np.log(
    df_clean['gold_oil_ratio'] / df_clean['gold_oil_ratio'].shift(1)
)

# Target remains forward 1-day log return
df_clean['Target_Log_Return'] = np.log(
    df_clean['gold close'].shift(-1) / df_clean['gold close']
)

# Clean missing rows created by 20-day rolling windows and shifts
df_enhanced = df_clean.dropna().reset_index(drop=True)

In [15]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# ---------------------------------------------------------
# 1. Feature & Target Selection
# ---------------------------------------------------------
# Identify numeric and categorical feature lists
numeric_features = [
    c
    for c in df_clean.columns
    if c.endswith(('_ret', '_log_ret', '_1d', '_2d', '_5d', '_10d', '_volatility_5d', '_volatility_20d', '_sma20_dist', '_chg', '_sin', '_cos'))
]

categorical_features = ['DayOfWeek', 'IsMonthEnd']
features = numeric_features + categorical_features
target = 'Target_Log_Return'

# Drop missing values from shifts/rolling windows
df_model = df_clean.dropna(subset=features + [target]).reset_index(drop=True)

# Chronological Train / Test Split (80/20)
split_idx = int(len(df_model) * 0.80)

X_train = df_model.iloc[:split_idx][features]
y_train = df_model.iloc[:split_idx][target]

X_test = df_model.iloc[split_idx:][features]
y_test = df_model.iloc[split_idx:][target]

# ---------------------------------------------------------
# 2. Preprocessing Pipeline
# ---------------------------------------------------------
numeric_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

tscv = TimeSeriesSplit(n_splits=5)

In [16]:
# Create Pipeline
rf_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

# Parameter grid tuned to prevent overfitting on return noise
rf_param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [3, 5, 8],
    'rf__min_samples_leaf': [5, 10, 20],
    'rf__max_features': ['sqrt', 0.5],
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
print('Best Random Forest Params:', rf_search.best_params_)

Best Random Forest Params: {'rf__max_depth': 3, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 5, 'rf__n_estimators': 200}


In [17]:
# Create Pipeline
xgb_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        (
            'xgb',
            XGBRegressor(
                random_state=42, objective='reg:squarederror', n_jobs=-1
            ),
        ),
    ]
)

# Parameter grid tuned for noisy time series data
xgb_param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [2, 3, 5],
    'xgb__learning_rate': [0.01, 0.05],
    'xgb__subsample': [0.7, 0.9],
    'xgb__colsample_bytree': [0.7, 0.9],
    'xgb__reg_alpha': [0.1, 1.0, 10.0],  # L1 Regularization
    'xgb__reg_lambda': [1.0, 10.0],  # L2 Regularization
}

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_
print('Best XGBoost Params:', xgb_search.best_params_)

Best XGBoost Params: {'xgb__colsample_bytree': 0.7, 'xgb__learning_rate': 0.01, 'xgb__max_depth': 2, 'xgb__n_estimators': 100, 'xgb__reg_alpha': 1.0, 'xgb__reg_lambda': 1.0, 'xgb__subsample': 0.7}


In [18]:
# Generate Predictions
y_pred_rf = best_rf.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)
y_pred_baseline = np.zeros_like(y_test)

# Calculate Metrics
results = pd.DataFrame(
    {
        'Model': ['Zero Baseline', 'Random Forest', 'XGBoost'],
        'Test MSE': [
            mean_squared_error(y_test, y_pred_baseline),
            mean_squared_error(y_test, y_pred_rf),
            mean_squared_error(y_test, y_pred_xgb),
        ],
        'Test RMSE': [
            np.sqrt(mean_squared_error(y_test, y_pred_baseline)),
            np.sqrt(mean_squared_error(y_test, y_pred_rf)),
            np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        ],
        'Test R2': [
            r2_score(y_test, y_pred_baseline),
            r2_score(y_test, y_pred_rf),
            r2_score(y_test, y_pred_xgb),
        ],
    }
)

print('\nModel Comparison on Holdout Test Set:')
print(results.to_string(index=False))


Model Comparison on Holdout Test Set:
        Model  Test MSE  Test RMSE   Test R2
Zero Baseline   0.00008   0.008962 -0.003041
Random Forest   0.00008   0.008941  0.001735
      XGBoost   0.00008   0.008954 -0.001333


In [19]:
# Extract feature names after OneHotEncoder transformation
feature_names = best_xgb['preprocessor'].get_feature_names_out()

# XGBoost Feature Importances
xgb_importances = best_xgb['xgb'].feature_importances_

importances_df = (
    pd.DataFrame({'feature': feature_names, 'importance': xgb_importances})
    .sort_values(by='importance', ascending=False)
    .reset_index(drop=True)
)

print('\nTop 10 Most Important Features (XGBoost):')
print(importances_df.head(10))


Top 10 Most Important Features (XGBoost):
                        feature  importance
0       num__gold_volatility_5d    0.821904
1     num__silver_close_log_ret    0.178096
2    num__oil_close_log_ret_10d    0.000000
3   num__sp500_close_log_ret_1d    0.000000
4   num__sp500_close_log_ret_2d    0.000000
5   num__sp500_close_log_ret_5d    0.000000
6  num__sp500_close_log_ret_10d    0.000000
7  num__nasdaq_close_log_ret_1d    0.000000
8  num__nasdaq_close_log_ret_2d    0.000000
9  num__nasdaq_close_log_ret_5d    0.000000


In [20]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. Construct Binary Directional Target
# ---------------------------------------------------------
# 1 if next day return is positive, 0 if negative/flat
df_model['Target_Direction'] = (df_model['Target_Log_Return'] > 0).astype(int)

y_train_dir = df_model.iloc[:split_idx]['Target_Direction']
y_test_dir = df_model.iloc[split_idx:]['Target_Direction']

# Baseline random guess or majority class probability
majority_class_acc = max(y_test_dir.mean(), 1 - y_test_dir.mean())

# ---------------------------------------------------------
# 2. Random Forest Classifier Pipeline
# ---------------------------------------------------------
rf_clf_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        (
            'rf',
            RandomForestClassifier(
                random_state=42, n_jobs=-1, class_weight='balanced'
            ),
        ),
    ]
)

rf_clf_params = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [2, 3, 5],
    'rf__min_samples_leaf': [10, 20, 50],
}

rf_clf_search = GridSearchCV(
    estimator=rf_clf_pipeline,
    param_grid=rf_clf_params,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
)

rf_clf_search.fit(X_train, y_train_dir)
best_rf_clf = rf_clf_search.best_estimator_

# ---------------------------------------------------------
# 3. XGBoost Classifier Pipeline
# ---------------------------------------------------------
xgb_clf_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        (
            'xgb',
            XGBClassifier(
                random_state=42, eval_metric='logloss', n_jobs=-1
            ),
        ),
    ]
)

xgb_clf_params = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [2, 3],
    'xgb__learning_rate': [0.01, 0.03],
    'xgb__subsample': [0.7, 0.8],
    'xgb__colsample_bytree': [0.7, 0.8],
    'xgb__reg_alpha': [1.0, 5.0],
}

xgb_clf_search = GridSearchCV(
    estimator=xgb_clf_pipeline,
    param_grid=xgb_clf_params,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
)

xgb_clf_search.fit(X_train, y_train_dir)
best_xgb_clf = xgb_clf_search.best_estimator_

# ---------------------------------------------------------
# 4. Evaluate Test Accuracy
# ---------------------------------------------------------
rf_preds = best_rf_clf.predict(X_test)
xgb_preds = best_xgb_clf.predict(X_test)

print(f'Majority Class Baseline Accuracy: {majority_class_acc * 100:.2f}%')
print(
    f'Random Forest Test Accuracy:     {accuracy_score(y_test_dir, rf_preds) * 100:.2f}%'
)
print(
    f'XGBoost Test Accuracy:           {accuracy_score(y_test_dir, xgb_preds) * 100:.2f}%'
)

# ---------------------------------------------------------
# 5. Simple Strategy Backtest (Long/Short Strategy Return)
# ---------------------------------------------------------
# Convert 0/1 predictions to +1/-1 signal
rf_signal = np.where(rf_preds == 1, 1, -1)
actual_log_returns = y_test.values

# Strategy daily returns = Signal * Actual Log Return
strategy_returns = rf_signal * actual_log_returns
cumulative_strategy_return = np.exp(np.cumsum(strategy_returns)) - 1
cumulative_buy_hold_return = np.exp(np.cumsum(actual_log_returns)) - 1

print(f'\nHoldout Cumulative Buy & Hold Return: {cumulative_buy_hold_return[-1] * 100:.2f}%')
print(f'Holdout Cumulative Strategy Return:   {cumulative_strategy_return[-1] * 100:.2f}%')

Majority Class Baseline Accuracy: 51.89%
Random Forest Test Accuracy:     52.57%
XGBoost Test Accuracy:           52.84%

Holdout Cumulative Buy & Hold Return: 44.08%
Holdout Cumulative Strategy Return:   46.67%


In [21]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---------------------------------------------------------
# 1. Pipeline Setup with ANOVA Feature Selection
# ---------------------------------------------------------
numeric_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

tscv = TimeSeriesSplit(n_splits=5)

# ---------------------------------------------------------
# 2. LightGBM + ANOVA Feature Selection Pipeline
# ---------------------------------------------------------
lgbm_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('anova', SelectKBest(score_func=f_classif)),  # ANOVA F-Test Step
        (
            'lgbm',
            LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1),
        ),
    ]
)

lgbm_param_grid = {
    'anova__k': [8, 12, 15, 'all'],  # Tune top ANOVA features to keep
    'lgbm__n_estimators': [50, 100],
    'lgbm__max_depth': [2, 3],
    'lgbm__learning_rate': [0.01, 0.03],
    'lgbm__num_leaves': [4, 7],
    'lgbm__reg_alpha': [0.1, 1.0, 10.0],
}

lgbm_search = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=lgbm_param_grid,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
)

lgbm_search.fit(X_train, y_train_dir)
best_lgbm = lgbm_search.best_estimator_

# ---------------------------------------------------------
# 3. Logistic Regression (ElasticNet) + ANOVA Pipeline
# ---------------------------------------------------------
lr_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('anova', SelectKBest(score_func=f_classif)),
        (
            'lr',
            LogisticRegression(
                solver='saga', penalty='elasticnet', random_state=42
            ),
        ),
    ]
)

lr_param_grid = {
    'anova__k': [5, 10, 15],
    'lr__l1_ratio': [0.1, 0.5, 0.7],  # Balance between L1 (Lasso) & L2 (Ridge)
    'lr__C': [0.001, 0.01, 0.1, 1.0],
}

lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
)

lr_search.fit(X_train, y_train_dir)
best_lr = lr_search.best_estimator_

# ---------------------------------------------------------
# 4. Out-of-Sample Evaluation
# ---------------------------------------------------------
lgbm_preds = best_lgbm.predict(X_test)
lr_preds = best_lr.predict(X_test)

lgbm_probs = best_lgbm.predict_proba(X_test)[:, 1]
lr_probs = best_lr.predict_proba(X_test)[:, 1]

print('--- Model Comparison ---')
print(f'Best ANOVA k (LightGBM): {best_lgbm["anova"].k}')
print(
    f'LightGBM Test Accuracy:   {accuracy_score(y_test_dir, lgbm_preds) * 100:.2f}% | ROC-AUC: {roc_auc_score(y_test_dir, lgbm_probs):.4f}'
)
print(f'Best ANOVA k (ElasticNet): {best_lr["anova"].k}')
print(
    f'ElasticNet Test Accuracy: {accuracy_score(y_test_dir, lr_preds) * 100:.2f}% | ROC-AUC: {roc_auc_score(y_test_dir, lr_probs):.4f}'
)

/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with featur

--- Model Comparison ---
Best ANOVA k (LightGBM): all
LightGBM Test Accuracy:   52.16% | ROC-AUC: 0.5204
Best ANOVA k (ElasticNet): 5
ElasticNet Test Accuracy: 51.89% | ROC-AUC: 0.5000


/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [24]:
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

# Silence Optuna logging noise
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ---------------------------------------------------------
# 1. Define Optuna Objective Function for LightGBM
# ---------------------------------------------------------
def objective(trial):
    # Hyperparameter search space
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1,
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.005, 0.1, log=True
        ),
        'num_leaves': trial.suggest_int('num_leaves', 4, 31),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # Transform features in fold to prevent leakage
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train_dir.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train_dir.iloc[val_idx]

        # Fit preprocessor on fold train set
        X_tr_proc = preprocessor.fit_transform(X_tr)
        X_va_proc = preprocessor.transform(X_va)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr_proc, y_tr)

        preds = model.predict_proba(X_va_proc)[:, 1]
        score = roc_auc_score(y_va, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)


# ---------------------------------------------------------
# 2. Run Optimization Study (50 Trials)
# ---------------------------------------------------------
print('Starting Optuna Bayesian Optimization for LightGBM...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('\n--- Optuna Study Results ---')
print(f'Best CV ROC-AUC: {study.best_value:.4f}')
print('Best Hyperparameters:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

# ---------------------------------------------------------
# 3. Train Final Model with Best Params & Evaluate on Test Set
# ---------------------------------------------------------
best_params = study.best_params
best_params.update(
    {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'random_state': 42,
        'verbose': -1,
    }
)

# Preprocess full training and test sets
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

opt_lgbm = lgb.LGBMClassifier(**best_params)
opt_lgbm.fit(X_train_proc, y_train_dir)

opt_preds = opt_lgbm.predict(X_test_proc)
opt_probs = opt_lgbm.predict_proba(X_test_proc)[:, 1]

print('\n--- Out-of-Sample Test Evaluation (Optuna LightGBM) ---')
print(f'Test Accuracy: {accuracy_score(y_test_dir, opt_preds) * 100:.2f}%')
print(f'Test ROC-AUC:  {roc_auc_score(y_test_dir, opt_probs):.4f}')

/Users/dan/tf_project_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Optuna Bayesian Optimization for LightGBM...


  0%|          | 0/50 [00:00<?, ?it/s]/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/dan/tf_project_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but


--- Optuna Study Results ---
Best CV ROC-AUC: 0.5100
Best Hyperparameters:
  learning_rate: 0.09672554956832971
  num_leaves: 22
  max_depth: 6
  min_child_samples: 100
  subsample: 0.5592125099849601
  colsample_bytree: 0.846945152449234
  reg_alpha: 0.21291800524694185
  reg_lambda: 6.353244250689875

--- Out-of-Sample Test Evaluation (Optuna LightGBM) ---
Test Accuracy: 51.76%
Test ROC-AUC:  0.5205
